# 03 — Relationships, Cardinality & Join Coverage

**Objetivo:** validar empíricamente las relaciones entre las siete fuentes antes de materializar joins.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Métricas
- **Coverage:** proporción de filas del lado izquierdo con match.
- **Multiplicity:** cuántas filas del lado derecho puede tener una key.
- **Expansion factor:** filas esperadas tras LEFT JOIN / filas originales.

Un join que ejecuta sin error no es necesariamente un join correcto.

In [ ]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

In [ ]:
from argos.profiling.relationships import analyze_all_relationships
from argos.common.reports import write_report

cfg = PROJECT_CONFIG["profiling"]
join_analysis = analyze_all_relationships(
    RELATIONSHIPS,
    FILES,
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    join_analysis,
    paths.reports / "profiling" / "join_analysis.parquet",
)

join_analysis[[
    "relationship", "status", "left_dataset", "right_dataset",
    "left_row_coverage", "max_right_multiplicity", "expansion_factor"
]]

In [ ]:
# Relaciones que requieren investigación antes de aceptarse.
join_analysis.loc[
    (join_analysis["status"] != "analyzed")
    | (join_analysis["left_row_coverage"].fillna(0) < 0.95)
    | (join_analysis["expansion_factor"].fillna(1) > 1.05)
]

## Criterio de salida
`docs/data-model.md` debe distinguir **relaciones verificadas** de **hipótesis todavía no aceptadas**.
No construyas una mega-tabla en Bronze.